<a href="https://colab.research.google.com/github/NguyenThien1906/Image-Video-Processing/blob/main/src/Functions_and_classes_REDESIGN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
CALC_FLAG = 'cpu'
# 'cpu' or 'cuda'

# Functions

## CPU-jit functions

In [13]:
from numba import jit, prange
import numpy as np

@jit(nopython=True)
def trunc(val, min, max):
  if val < min:
    return min
  elif val > max:
    return max
  return val

### N-to-1 functions

In [2]:
@jit(nopython=True, parallel=True)
def mean_cpu(image2d):
  ans = 0
  for x in prange(image2d.shape[0]):
    for y in prange(image2d.shape[1]):
      ans += image2d[x, y]
  ans /= (image2d.shape[0] * image2d.shape[1])
  return ans

@jit(nopython=True, parallel=True)
def var_cpu(image2d):
  Mean = mean_cpu(image2d)
  L2sum = 0

  for y in prange(image2d.shape[1]):
    for x in prange(image2d.shape[0]):
      L2sum += (image2d[x,y] - Mean) ** 2
  return L2sum / image2d.shape[0] / image2d.shape[1]

In [8]:
image2d_dummy = np.random.rand(1, 1)

mean_cpu(image2d_dummy)
var_cpu(image2d_dummy)

0.0

### 1-to-1 functions

In [3]:
@jit(nopython=True, parallel=True)
def add_cpu(input4d_1, input4d_2, output4d):
  for k in prange(input4d_1.shape[0]):
    for c in prange(input4d_1.shape[1]):
      for x in prange(input4d_1.shape[2]):
        for y in prange(input4d_1.shape[3]):
          output4d[k,c,x,y] = input4d_1[k,c,x,y] + input4d_2[k,c,x,y]

@jit(nopython=True, parallel=True)
def bn_cpu(input4d, output4d, stab):
  channels = input4d.shape[1]
  for k in prange(input4d.shape[0]):
    for c in prange(input4d.shape[1]):
      Mean = mean_cpu(input4d[k,c])
      Var = var_cpu(input4d[k,c])

      for x in prange(input4d.shape[2]):
        for y in prange(input4d.shape[3]):
          output4d[k,c,x,y] = (input4d[k,c,x,y] - Mean) / np.sqrt(Var + stab)

@jit(nopython=True)
def leakyrelu_cpu(input4d, output4d, alpha):
  for i in prange(input4d.shape[0]):
    for j in prange(input4d.shape[1]):
      for k in prange(input4d.shape[2]):
        for l in prange(input4d.shape[3]):
          if input4d[i,j,k,l] < 0:
            output4d[i,j,k,l] = alpha * input4d[i,j,k,l]
          else:
            output4d[i,j,k,l] = input4d[i,j,k,l]

@jit(nopython=True, parallel=True)
def softmax_cpu(input2d, output2d):
  for i in prange(output2d.shape[0]):
    sum = 0
    for j in prange(input2d.shape[1]):
      temp = np.exp(input2d[i,j])
      output2d[i,j] = temp # for GPU GMEM stuff
      sum += temp
    for j in prange(input2d.shape[1]):
      output2d[i,j] = output2d[i,j] / sum

In [10]:
input4d_dummy = np.random.rand(1, 1, 1, 1)
output4d_dummy = np.zeros_like(input4d_dummy)

add_cpu(input4d_dummy, input4d_dummy, output4d_dummy)
bn_cpu(input4d_dummy, output4d_dummy, 1e-5)
leakyrelu_cpu(input4d_dummy, output4d_dummy, 0.01)
softmax_cpu(input4d_dummy[0,0], output4d_dummy[0,0])

### N-to-N functions

In [14]:
@jit(nopython=True, parallel=True)
def conv2d_cpu(input4d, kernel3d, bias1d, output4d, stride, padding, dilation):
  for im in prange(output4d.shape[0]):
    for c_out in prange(output4d.shape[1]):
      for x in prange(output4d.shape[2]):
        for y in prange(output4d.shape[3]):
          sum = 0
          for x_k in prange(kernel3d.shape[1]):
            for y_k in prange(kernel3d.shape[2]):
              x_in = x * stride + x_k * dilation - padding
              y_in = y * stride + y_k * dilation - padding
              x_in = trunc(x_in, 0, input4d.shape[2] - 1)
              y_in = trunc(y_in, 0, input4d.shape[3] - 1)

              sum_pi = 0
              for c_in in prange(input4d.shape[1]):
                sum_pi += input4d[im, c_in, x_in, y_in]
              sum += sum_pi * kernel3d[c_out, x_k, y_k]
          output4d[im, c_out, x, y] = sum + bias1d[c_out]

@jit(nopython=True, parallel=True)
def global_avgpool2d_scale_cpu(input4d, output4d, scale_factor):
  for im in prange(output4d.shape[0]):
    for c_out in prange(output4d.shape[1]):
      for x in prange(output4d.shape[2]):
        for y in prange(output4d.shape[3]):
          sum = 0
          for x_k in prange(scale_factor):
            for y_k in prange(scale_factor):
              x_in = trunc(x * scale_factor + x_k, 0, input4d.shape[2] - 1)
              y_in = trunc(y * scale_factor + y_k, 0, input4d.shape[3] - 1)
              sum += input4d[im, c_out, x_in, y_in]
          output4d[im, c_out, x, y] = sum / (scale_factor * scale_factor)

@jit(nopython=True, parallel=True)
def lineardense_cpu(input2d, weight2d, bias1d, output2d):
  for i in prange(output2d.shape[0]):
    for j in prange(output2d.shape[1]):
      sum = 0
      for k in prange(input2d.shape[1]):
        sum += input2d[i,k] * weight2d[k,j]
      output2d[i,j] = sum + bias1d[j]

@jit(nopython=True, parallel=True)
def local_maxpool2d_cpu(input4d, output4d, kernel_size, stride, padding, dilation):
  for im in prange(output4d.shape[0]):
    for c_out in prange(output4d.shape[1]):
      for x in prange(output4d.shape[2]):
        for y in prange(output4d.shape[3]):
          max_val = -np.inf
          for x_k in prange(kernel_size):
            for y_k in prange(kernel_size):
              x_in = x * stride + x_k * dilation - padding
              y_in = y * stride + y_k * dilation - padding
              x_in = trunc(x_in, 0, input4d.shape[2] - 1)
              y_in = trunc(y_in, 0, input4d.shape[3] - 1)
              if input4d[im, c_out, x_in, y_in] > max_val:
                max_val = input4d[im, c_out, x_in, y_in]
          output4d[im, c_out, x, y] = max_val

In [15]:
input4d_dummy = np.random.rand(1, 1, 1, 1)
kernel3d_dummy = np.random.rand(1, 1, 1)
bias1d_dummy = np.random.rand(1)
output4d_dummy = np.zeros_like(input4d_dummy)

conv2d_cpu(input4d_dummy, kernel3d_dummy, bias1d_dummy, output4d_dummy, 1, 0, 1)
global_avgpool2d_scale_cpu(input4d_dummy, output4d_dummy, 1)
lineardense_cpu(input4d_dummy[0,0], kernel3d_dummy[0], bias1d_dummy, output4d_dummy[0])
local_maxpool2d_cpu(input4d_dummy, output4d_dummy, 1, 1, 0, 1)

## GPU functions

## Function central

Backbone functions need to be included in a specific class to allow checking the time and resources used to calculate. Note: we cannot use decorators on CUDA functions, so these functions are treated as mediums instead.

We use Singleton design pattern for this architecture.

In [18]:
import torch
import numpy as np
import math

# macros
CPU_FLAG = 'cpu'
GPU_FLAG = 'cuda'
CUDA_FLAG = torch.cuda.is_available()

class BackFunctions:
    _instance = None
    GLOBAL_DTYPE = np.float32

    # simple functions with no need for parallelization=======================
    @classmethod
    def trunc(val, min, max):
      if val < min:
        return min
      elif val > max:
        return max
      return val

    # 1-to-1 functions========================================================
    @classmethod
    def add(image4d_1, image4d_2, output4d):
      output4d = np.zeros_like(image4d_1)
      if CALC_FLAG == CPU_FLAG:
        add_cpu(image4d_1, image4d_2, output4d)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d

    @classmethod
    def bn(image4d, stab=1e-5):
      output4d = np.zeros_like(image4d)

      if CALC_FLAG == CPU_FLAG:
        bn_cpu(image4d, output4d, stab)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d

    @classmethod
    def leakyrelu(image4d, alpha=0.01):
      output4d = np.zeros_like(image4d)

      if CALC_FLAG == CPU_FLAG:
        leakyrelu_cpu(image4d, output4d, alpha)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d

    @classmethod
    def softmax(image2d):
      output2d = np.zeros_like(image2d)

      if CALC_FLAG == CPU_FLAG:
        softmax_cpu(image2d, output2d)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output2d

    # n-to-1 functions========================================================
    @classmethod
    def mean(image2d):
      ans = 0

      if CALC_FLAG == CPU_FLAG:
        ans = mean_cpu(image2d)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return ans

    @classmethod
    def var(image2d):
      ans = 0
      if CALC_FLAG == CPU_FLAG:
        ans = var_cpu(image2d)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return ans

    # n-to-n functions========================================================
    @classmethod
    def conv2d(input4d, kernel3d, bias1d, output4d, stride=1, padding=1, dilation=1):
      # setup
      output_shape = (input4d.shape[0], kernel3d.shape[0],
                      (input4d.shape[2] - (kernel3d.shape[1] - 1) * dilation+ 2*padding- 1)//stride + 1,
                      (input4d.shape[3] - (kernel3d.shape[2] - 1) * dilation+ 2*padding- 1)//stride + 1)
      output4d = np.zeros(output_shape, dtype=BackFunctions.GLOBAL_DTYPE)

      # execution
      if CALC_FLAG == CPU_FLAG:
        conv2d_cpu(input4d, kernel3d, bias1d, output4d, stride, padding, dilation)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d

    @classmethod
    def global_avgpool2d_scale(input4d, output4d, scale_factor=1):
      # setup
      output_shape = (input4d.shape[0], input4d.shape[1],
                      input4d.shape[2] // scale_factor,
                      input4d.shape[3] // scale_factor)
      output4d = np.zeros(output_shape, dtype=BackFunctions.GLOBAL_DTYPE)

      # execution
      if CALC_FLAG == CPU_FLAG:
        global_avgpool2d_scale_cpu(input4d, output4d, scale_factor)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d

    @classmethod
    def lineardense(input2d, weight2d, bias1d, output2d):
      # setup
      output_shape = (input2d.shape[0], weight2d.shape[1])
      output2d = np.zeros(output_shape, dtype=BackFunctions.GLOBAL_DTYPE)

      # execution
      if CALC_FLAG == CPU_FLAG:
        lineardense_cpu(input2d, weight2d, bias1d, output2d)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output2d

    @classmethod
    def local_maxpool2d(input4d, output4d, kernel_size, stride, padding, dilation):
      # setup
      output_shape = (input4d.shape[0], input4d.shape[1],
                      (input4d.shape[2] - (kernel_size - 1) * dilation+ 2 * padding- 1)//stride + 1,
                      (input4d.shape[3] - (kernel_size - 1) * dilation+ 2 * padding- 1) // stride + 1)
      output4d = np.zeros(output_shape, dtype=BackFunctions.GLOBAL_DTYPE)

      # execution
      if CALC_FLAG == CPU_FLAG:
        local_maxpool2d_cpu(input4d, output4d, kernel_size, stride, padding, dilation)

      elif CALC_FLAG == GPU_FLAG and CUDA_FLAG:
        raise NotImplementedError

      return output4d


# YOLOv4 backbone: CSPDarknet53

In [ ]:
import math
import numpy as np

class NNLayer:
  def __init__(self, in_channels):
    self.in_channels = in_channels

  def forward(self, input):
    if len(input.shape) != 4:
      raise ValueError(f"Input has to be a 4D array, got {len(input.shape)} instead.")
    elif input.shape[1] != self.in_channels:
      raise ValueError(f"Input has to have {self.in_channels} channels, got {input.shape[1]} instead.")

class Conv2D(NNLayer):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)
    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

    self.kernel = np.random.randn(out_channels, kernel_size, kernel_size)
    self.bias = np.random.randn(out_channels)

  def forward(self, input):
    # check
    super().forward(input)

    # execute
    output = BackFunctions.conv2d(input, self.kernel, self.bias, self.stride, self.padding, self.dilation)
    output = BackFunctions.bn(output)
    output = BackFunctions.leakyrelu(output)

    return output

class CSPResidualBlock(NNLayer):
  def __init__(self, in_channels):
    if in_channels % 4 != 0:
      raise ValueError("in_channels must be divisible by 4")
    super().__init__(in_channels)

    split_channels = in_channels // 2
    self.conv1 = Conv2D(split_channels, split_channels // 2, 1, 1, 0, 1)
    self.conv2 = Conv2D(split_channels // 2, split_channels, 3, 1, 1, 1)
    self.conv3 = Conv2D(in_channels, in_channels, 1, 1, 0, 1)

  def forward(self, input):
    # check
    super().forward(input)

    # execute
    input1, input2 = np.split(input, 2, axis=1)
    input1 = self.conv1.forward(input1)
    input1 = self.conv2.forward(input1)
    output = np.concatenate([input1, input2], axis=1)
    output = self.conv3.forward(output)

    return output

class NN:
  def __init__(self, layers = None):
    if layers is None:
      self.layers = []
    elif isinstance(layers, NNLayer):
      self.layers = [layers]
    elif isinstance(layers, list) and ((len(layers) > 0 and all([])) or len(layers) == 0):
      self.layers = layers
    else:
      raise TypeError("Unsupported layer type")

  def forward(self, input):
    output = input
    for i in range(len(self.layers)):
      output = self.layers[i].forward(output)
    return output

In [ ]:
class CSPDarknet53:
  def __init__(self, in_channels):
    self.in_channels = in_channels
    self.csp_p3 = NN([
        Conv2D(in_channels, 32, 3, 1, 1, 1),
        Conv2D(32, 64, 3, 2, 1, 1),
        CSPResidualBlock(64),
        Conv2D(64, 128, 3, 2, 1, 1),
        CSPResidualBlock(128),
        CSPResidualBlock(128),
        Conv2D(128, 256, 3, 2, 1, 1),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
        CSPResidualBlock(256),
    ])

    self.csp_p4 = NN([
        Conv2D(256, 512, 3, 2, 1, 1),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512),
        CSPResidualBlock(512)
    ])

    self.csp_p5 = NN([
        Conv2D(512, 1024, 3, 2, 1, 1),
        CSPResidualBlock(1024),
        CSPResidualBlock(1024),
        CSPResidualBlock(1024),
        CSPResidualBlock(1024)
    ])


  def forward(self, input):
    output = self.csp_p3.forward(input)
    output2 = self.csp_p4.forward(output)
    output3 = self.csp_p5.forward(output2)
    return output, output2, output3

# YOLOv4 neck: Spatial Pyramid Pooling (SPP)

In [ ]:
class LocalMaxPooling2d(NNLayer):
  def __init__(self, in_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

  def forward(self, input):
    super().forward(input)

    output = BackFunctions.local_maxpool2d(input, self.kernel_size, self.stride, self.padding, self.dilation)

    return output

class SPP(NNLayer):
  def __init__(self, in_channels):
    super().__init__(in_channels)
    self.maxpool5 = LocalMaxPooling2d(in_channels, 5, 1, 2, 1)
    self.maxpool9 = LocalMaxPooling2d(in_channels, 9, 1, 4, 1)
    self.maxpool13 = LocalMaxPooling2d(in_channels, 13, 1, 6, 1)
    self.conv = Conv2D(in_channels * 4, in_channels, 1, 1, 0, 1)

  def forward(self, input):
    super().forward(input)

    output1 = self.maxpool5.forward(input)
    output2 = self.maxpool9.forward(input)
    output3 = self.maxpool13.forward(input)

    output = np.concatenate([input, output1, output2, output3], axis=1)
    output = self.conv.forward(input)

    return output

# YOLOv4 neck: Path Aggregation Network (PAN)

# YOLOv4 head: YOLOv3 detection heads

In [ ]:
class DetectionHead(NNLayer):
  def __init__(self, in_channels, num_classes):
    super().__init__(in_channels)
    self.num_classes = num_classes